# Kubeflow Pipeline

Goal: rebuild the notebook workflow from `04-kf-train.ipynb` as a Kubeflow
Pipeline, one step at a time.

`04` runs everything inside a single long-lived notebook pod. A pipeline splits
the same work into containers that KFP schedules independently, so a step can
be cached, retried, or given its own CPU/memory request.

Target shape:

```txt
fetch_data -> prepare_data -> train -> evaluate -> upload_model
```

**This notebook covers the data processing steps only** - `fetch_data` and
`prepare_data`. Training comes next.

Run this from the `yolo-cpu` notebook in the cluster: `kfp.Client()` talks to
`ml-pipeline-ui`, a ClusterIP service behind istio, which is unreachable from a
laptop.

## Environment

Install the SDK.

In [ ]:
# pip install
%pip install -q -U kfp

`kfp.Client()` authenticates with a projected ServiceAccount token. The
notebook's default token carries the cluster audience and gets rejected, so the
`kfp-api-token` PodDefault mounts one scoped to `pipelines.kubeflow.org`.

If the token path is missing, apply the PodDefault and restart the notebook:

```sh
kubectl apply -f kubeflow/notebook/kfp-api-token.yaml
```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import kfp
from kfp import compiler, dsl
from kfp.client.set_volume_credentials import ServiceAccountTokenVolumeCredentials
from kfp.dsl import Dataset, Input, Output

# Repo root
ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
)

# AWS
BUCKET = os.environ.get("BUCKET", "kubeflow-yolo-dev-099139718958")
REGION = os.environ.get("AWS_REGION", "ca-central-1")

# Kubeflow
NAMESPACE = "kubeflow-user-example-com"
# the in-cluster service, not the public gateway: the gateway front-ends
# oauth2-proxy, which answers a token-authenticated call with a login page
HOST = "http://ml-pipeline-ui.kubeflow.svc.cluster.local"
TOKEN_PATH = "/var/run/secrets/kubeflow/pipelines/token"

# where compiled pipelines land
BUILD = ROOT / "kubeflow" / "pipelines"
BUILD.mkdir(parents=True, exist_ok=True)

print("python   ", sys.version.split()[0])
print("kfp      ", kfp.__version__)
print("root     ", ROOT)
print("bucket   ", BUCKET)
print("region   ", REGION)
print("token    ", Path(TOKEN_PATH).exists())

Connect to the pipelines API.

In [ ]:
if not Path(TOKEN_PATH).exists():
    raise RuntimeError(
        f"no ServiceAccount token at {TOKEN_PATH}. Apply "
        "kubeflow/notebook/kfp-api-token.yaml, confirm the notebook carries "
        'the `kfp-api-token: "true"` label, then restart the notebook.'
    )

# construct client
client = kfp.Client(
    host=HOST,
    credentials=ServiceAccountTokenVolumeCredentials(path=TOKEN_PATH),
)

# a healthy call proves both the token audience and the istio policy are right
print(client.list_experiments(namespace=NAMESPACE).total_size, "experiments")

### Dataset version

`data/raw.dvc` pins the dataset. Its `md5` is the hash of a `.dir` manifest in
S3 - a JSON list of `{md5, relpath}` for every file - so one hash addresses the
whole dataset.

Passing it as a *pipeline parameter* rather than hardcoding it means training
against new data is a parameter change, not a code change.

In [ ]:
import yaml

# read the pinned dataset version out of the dvc file
DVC_DIR_HASH = yaml.safe_load((ROOT / "data" / "raw.dvc").read_text())["outs"][0]["md5"]

print("dvc_dir_hash", DVC_DIR_HASH)

---

## Step 1: `fetch_data`

Pull the raw dataset out of S3 - the pipeline equivalent of the `download_raw`
cell in `04`.

A few things differ from the notebook version:

- **Output artifact.** The component declares `raw: Output[Dataset]`. KFP
  assigns it a path on the shared artifact store and passes that path to the
  next step. Steps never share a filesystem otherwise.
- **Imports go inside the function.** The body is serialized as source and runs
  in a fresh container; nothing from this notebook's scope exists there.
- **No credentials.** EKS Pod Identity attaches the S3 role to
  `default-editor`, which pipeline pods already run as, so `boto3` picks the
  credentials up automatically (`infra/60-s3-iam.tf`).

In [ ]:
# Pinned so the runtime does not move when the SDK changes its default.
BASE_IMAGE = "python:3.12"


@dsl.component(base_image=BASE_IMAGE, packages_to_install=["boto3"])
def fetch_data(
    bucket: str,
    dvc_dir_hash: str,
    region: str,
    raw: Output[Dataset],
):
    """Restore the DVC-tracked raw dataset from S3 into the `raw` artifact."""
    import json
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    import boto3

    s3 = boto3.client("s3", region_name=region)

    def dvc_key(md5: str) -> str:
        # DVC shards its content-addressed store by the first two hex chars.
        return "dvcstore/files/md5/" + md5[:2] + "/" + md5[2:]

    # the .dir object lists {"md5": ..., "relpath": ...} for every file
    manifest = json.loads(
        s3.get_object(Bucket=bucket, Key=dvc_key(dvc_dir_hash))["Body"].read()
    )

    root = Path(raw.path)
    root.mkdir(parents=True, exist_ok=True)

    def fetch(entry):
        target = root / entry["relpath"]
        target.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, dvc_key(entry["md5"]), str(target))

    # 1100+ small objects: this is latency-bound, not bandwidth-bound
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(fetch, manifest))

    images = [p for p in root.iterdir() if p.suffix.lower() in {".jpeg", ".jpg", ".png"}]
    if not images:
        raise RuntimeError("no images restored -- check the dvc_dir_hash")

    # metadata shows up on the artifact in the run UI
    raw.metadata["files"] = len(manifest)
    raw.metadata["images"] = len(images)
    raw.metadata["dvc_dir_hash"] = dvc_dir_hash

    print("fetched", len(manifest), "files /", len(images), "images to", root)

---

## Step 2: `prepare_data`

Split into train/val and write the ultralytics descriptor - the `build_split` +
`write_data_yaml` cells from `04`, collapsed into one component.

Two details that matter:

- **The split is seeded.** Without a fixed seed, two runs see different images
  and their mAP numbers are not comparable - you cannot tell whether a change
  helped or the split just got easier.
- **`data.yaml`'s `path` must be absolute.** Ultralytics resolves a relative
  dataset root against the process cwd, then against its own `DATASETS_DIR` -
  never against the yaml's own location.

In [ ]:
@dsl.component(base_image=BASE_IMAGE, packages_to_install=["pyyaml"])
def prepare_data(
    raw: Input[Dataset],
    val_fraction: float,
    split_seed: int,
    processed: Output[Dataset],
):
    """Build processed/{train,val}/{images,labels} plus data.yaml."""
    import random
    import shutil
    from pathlib import Path

    import yaml

    suffixes = {".jpeg", ".jpg", ".png"}
    src = Path(raw.path)
    dst = Path(processed.path)

    # images and labels pair by basename: foo.jpeg <-> foo.txt
    stems = sorted(p.stem for p in src.iterdir() if p.suffix.lower() in suffixes)
    random.Random(split_seed).shuffle(stems)
    cut = int(len(stems) * (1 - val_fraction))
    train_stems, val_stems = stems[:cut], stems[cut:]

    counts = {}
    for split, names in (("train", train_stems), ("val", val_stems)):
        for sub in ("images", "labels"):
            (dst / split / sub).mkdir(parents=True, exist_ok=True)
        for stem in names:
            image = next(p for p in src.glob(stem + ".*") if p.suffix.lower() in suffixes)
            shutil.copy(image, dst / split / "images" / image.name)
            label = src / (stem + ".txt")
            # an image with no label file is a legitimate negative sample
            if label.exists():
                shutil.copy(label, dst / split / "labels" / label.name)
        counts[split] = len(names)

    if not counts["train"] or not counts["val"]:
        raise RuntimeError("empty split: " + str(counts))

    class_names = (src / "classes.txt").read_text().split()
    (dst / "data.yaml").write_text(
        yaml.safe_dump(
            {
                "path": str(dst),          # absolute, see above
                "train": "train/images",
                "val": "val/images",
                "nc": len(class_names),
                "names": class_names,
            },
            sort_keys=False,
        )
    )

    processed.metadata.update(counts)
    processed.metadata["classes"] = class_names
    print("split", counts, "classes", class_names)
    print((dst / "data.yaml").read_text())

---

## Pipeline

Wire the two steps together. `raw=fetch.outputs["raw"]` is what creates the
dependency - KFP derives the DAG from the data flow, there is no explicit
ordering.

`fetch_data` is cached, so re-running with the same `dvc_dir_hash` skips the
download and starts at the split. That is the payoff for making the hash a
parameter: the cache key changes exactly when the dataset does.

In [ ]:
@dsl.pipeline(
    name="yolo-data",
    description="Fetch the DVC-tracked dataset from S3 and build the train/val split.",
)
def data_pipeline(
    bucket: str = BUCKET,
    dvc_dir_hash: str = DVC_DIR_HASH,
    region: str = REGION,
    val_fraction: float = 0.2,
    split_seed: int = 0,
):
    fetch = fetch_data(
        bucket=bucket,
        dvc_dir_hash=dvc_dir_hash,
        region=region,
    ).set_caching_options(True)

    prepare_data(
        raw=fetch.outputs["raw"],
        val_fraction=val_fraction,
        split_seed=split_seed,
    )

Compile. The generated yaml is what the KFP API consumes - it is the record of
exactly what a run executed, which is why it is worth committing.

In [ ]:
PACKAGE = BUILD / "data_pipeline.yaml"

# compile pipeline
compiler.Compiler().compile(data_pipeline, str(PACKAGE))

print("compiled", PACKAGE, f"({PACKAGE.stat().st_size / 1e3:.1f} kB)")

## Submit

Each submit creates a run under the `yolo-data` experiment.

In [ ]:
EXPERIMENT = "yolo-data"

# reuses the experiment if it already exists
experiment = client.create_experiment(name=EXPERIMENT, namespace=NAMESPACE)

# submit run
run = client.run_pipeline(
    experiment_id=experiment.experiment_id,
    job_name="data-split",
    pipeline_package_path=str(PACKAGE),
    arguments={},  # everything keeps its pipeline default
)

print("run", run.run_id)
print("ui ", "/pipeline/#/runs/details/" + run.run_id)

Block until it finishes. The fetch pulls ~200 MB across 1113 objects, so a cold
run takes a few minutes; a cached one returns almost immediately.

In [ ]:
# wait for run
result = client.wait_for_run_completion(run.run_id, timeout=1800)

print("state   ", result.state)
print("duration", result.finished_at - result.created_at)

### Inspect

Pull the step outputs back out of the API. `processed.metadata` carries the
train/val counts the component wrote, which is the quickest confirmation the
split landed as expected.

If a step failed, its logs are the fastest way in:

```sh
kubectl get workflows -n kubeflow-user-example-com
kubectl logs -n kubeflow-user-example-com <pod> -c main
```

In [ ]:
import json

# get run detail
detail = client.get_run(run.run_id)

for task in detail.run_details.task_details or []:
    # the DAG root has no outputs of its own
    if not task.outputs:
        continue

    print(f"{task.display_name:14} {task.state}")
    for key, artifact_list in (task.outputs.artifacts or {}).items():
        for artifact in artifact_list.artifacts:
            print(f"  {key}  {artifact.uri}")
            print("  " + json.dumps(dict(artifact.metadata), indent=2, default=str))
    print()

---

## Next

- `train` - fine-tune `yolo11n.pt` on `processed`, emit `best.pt` as an
  `Output[Model]`.
- `evaluate` - re-validate `best.pt` and log mAP to the run's Metrics tab.
- `upload_model` - copy the weights to `s3://<bucket>/models/<run-id>/`.